In [38]:
import copy
import os

import h5py
import numpy as np
import matplotlib.pyplot as plt
########################################

DIRPATH = os.getcwd()
PLOT_DIRPATH = os.path.join(DIRPATH, 'v10/Probs')
if not os.path.exists(PLOT_DIRPATH): os.makedirs(PLOT_DIRPATH)

In [ ]:
pred_f = h5py.File('/storage/af/user/tsievert/topNet/tt_hadronic_bqfixed.h5', 'r')     # prediction file (model output)
truth_f = h5py.File('/storage/af/user/tsievert/topNet/tt_hadronic_fixed_test.h5', 'r')  # truth/test file (has MASK)

In [ ]:
recos = sorted(truth_f['TARGETS'].keys())
pred_dict = {
    reco: {
        key: pred_f['SpecialKey.Targets'][reco][key][:] for key in pred_f['SpecialKey.Targets'][reco].keys()
    } for reco in recos
}
targ_dict = {
    reco: {
        key: truth_f['TARGETS'][reco][key][:] for key in truth_f['TARGETS'][reco].keys()
    } for reco in recos
}
for FRreco in [reco for reco in recos if 'FR' in reco]:  # For q1/2 assignment symmetry
    FRrecoalt = FRreco+'_alt'
    targ_dict[FRrecoalt] = copy.deepcopy(targ_dict[FRreco])
    targ_dict[FRrecoalt]['swap'] = copy.deepcopy(targ_dict[FRrecoalt]['q1'])
    targ_dict[FRrecoalt]['q1'] = copy.deepcopy(targ_dict[FRrecoalt]['q2'])
    targ_dict[FRrecoalt]['q2'] = copy.deepcopy(targ_dict[FRrecoalt]['swap'])
    del targ_dict[FRrecoalt]['swap']


In [39]:
def plot_probs(dps, aps, dpaps, labels, mode: str, title_extra: str='', file_postfix: str=''):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Sample (unmasked) plots
    axes[0].hist(dps, bins=50, range=(0, 1), stacked=True, label=labels)
    axes[0].set_xlabel('Detection Probability')
    axes[0].set_ylabel('Events')
    axes[0].set_title(f'{mode} Detection Probability')
    axes[0].set_yscale('log')

    axes[1].hist(aps, bins=50, range=(0, 1), stacked=True, label=labels)
    axes[1].set_xlabel('Assignment Probability')
    axes[1].set_ylabel('Events')
    axes[1].set_title(f'{mode} Assignment Probability')
    axes[1].set_yscale('log')

    axes[2].hist(dpaps, bins=50, range=(0, 1), stacked=True, label=labels)
    axes[2].set_xlabel('Detection * Assignment Probability')
    axes[2].set_ylabel('Events')
    axes[2].set_title(f'{mode} Detection * Assignment Probability')
    axes[2].set_yscale('log')

    plt.suptitle(f"{mode} {title_extra} Probabilities", fontsize=13)
    plt.tight_layout()
    plt.legend()
    plt.savefig(os.path.join(PLOT_DIRPATH, f"{mode}_ProbHist{file_postfix}.png"), dpi=150)
    plt.close()

In [23]:
def correct_mask(pred_idxs, targ_idxs, valid: bool=True):
    correct_assn = targ_idxs['MASK'] if valid else ~targ_idxs['MASK']
    for pred_idx in [key for key in pred_idxs.keys() if 'prob' not in key]:
        correct_assn = np.logical_and(
            correct_assn, 
            (pred_idxs[pred_idx] == targ_idxs[pred_idx]) 
            if valid else ~(pred_idxs[pred_idx] == targ_idxs[pred_idx])
        )
    return correct_assn


In [ ]:
# Only look at events where FRt1 is actually the valid/applicable target
valid_ps = {reco.replace('t1', ''): copy.deepcopy({'dp': list(), 'ap': list(), 'dpap': list(), 'label': list()}) for reco in recos if 't1' in reco}
invalid_ps = copy.deepcopy(valid_ps)
for reco in recos:
    pskey = [key for key in valid_ps.keys() if key in reco][0]

    # Valid targets
    corr_assn = np.zeros_like(pred_dict[reco][list(pred_dict[reco].keys())[0]], dtype=bool)
    for targ in [key for key in targ_dict.keys() if reco in key]:
        corr_assn = np.logical_or(
            corr_assn, correct_mask(pred_dict[reco], targ_dict[targ])
        )
    for label, value in zip(['corr', 'incorr'], [True, False]):
        valid_ps[pskey]['dp'].append(pred_dict[reco]['detection_probability'][corr_assn == value])
        valid_ps[pskey]['ap'].append(pred_dict[reco]['assignment_probability'][corr_assn == value])
        valid_ps[pskey]['dpap'].append(valid_ps[pskey]['dp'][-1] * valid_ps[pskey]['ap'][-1])
        valid_ps[pskey]['label'].append(reco+' '+label)

    # Invalid targets
    corr_assn = correct_mask(pred_dict[reco], targ_dict[reco], valid=False)
    invalid_ps[pskey]['dp'].append(pred_dict[reco]['detection_probability'][corr_assn])
    invalid_ps[pskey]['ap'].append(pred_dict[reco]['assignment_probability'][corr_assn])
    invalid_ps[pskey]['dpap'].append(invalid_ps[pskey]['dp'][-1] * invalid_ps[pskey]['ap'][-1])
    invalid_ps[pskey]['label'].append(reco+' '+label)


In [40]:
for key, probs in valid_ps.items():
    plot_probs(probs['dp'], probs['ap'], probs['dpap'], probs['label'], key, title_extra='Valid Targets', file_postfix='_validT')

for key, probs in invalid_ps.items():
    plot_probs(probs['dp'], probs['ap'], probs['dpap'], probs['label'], key, title_extra='Invalid Targets', file_postfix='_invalidT')